# 在数中间填入运算符凑出 99：由 Z3 决定括号的位置

## 这一章讲什么

数的顺序是固定的，只能在中间填入 `+`, `-`, `*` 这三个运算符，再靠括号改变计算的顺序。这一章让 Z3 代替我们挑选运算符与括号的位置，最后把找到的步骤还原成可以阅读的算式。

- 反复做“把相邻两个数算出来合成一个”，任何括号方式都能这样表示
- 把每个阶段的数组设成 Z3 变量，用 `If` 挑选运算
- 最后再把找到的步骤还原成算式字符串

此外还能看清它与数的顺序自由的“凑出 24”之间差在什么地方：后者不但数的顺序可变，还要多补一条排列约束。

> 下面的式子里，每个 `?` 都填入 `+`, `-`, `*` 中的一个，括号可以任意摆放，使式子的值正好等于 99。
>
> `1 ? 2 ? 3 ? 4 ? 5 ? 6 ? 7 ? 8 ? 9 = 99`

## 思路

我们不对每一个数顺着往下计算，而是反复做同一个动作：把相邻的两个数算出来，合成一个。例如从 `1 2 3 4 5` 出发，依次做下面四步，其中每一步都表示把两个数替换成一个，数的个数也随之减少一个。

1. 计算 `3 + 4` → `1 2 7 5`
2. 计算 `1 * 2` → `2 7 5`
3. 计算 `7 + 5` → `2 12`
4. 计算 `2 - 12` → `-10`

最后只剩下一个数，那个数就是式子的值。

```{mermaid}
graph
a1[1]
a2[2]
a3[3+4]
a4[5]
b1[1*2]
b2[3+4]
b3[5]
c1[1*2]
c2[3+4-5]
d1[1*2+3+4-5]
1 --> a1
2 --> a2
3 --> a3
4 --> a3
5 --> a4
a1 --> b1
a2 --> b1
a3 --> b2
a4 --> b3
b1 --> c1
b2 --> c2
b3 --> c2
c1 --> d1
c2 --> d1
```

至于每一步在哪个位置合并、使用哪一个运算符，全部交给 Z3 决定。这么一来，等于把括号的每一种摆法都试过一遍；阶段一共 $n-1$ 个，每个阶段能挑选的位置是“当时剩下的元素个数减 1”个。

## 一段一段地计算

阶段 `l` 的数组记作 `f[l]`，其中开头的 `f[0]` 是 `[1, 2, …, 9]`，越往后走越短，每个阶段比前一阶段少一个元素。

`calc_tree(f, locs, ops)` 所做的工作，就是把每个阶段的计算逐段写成约束。在阶段 `l` 选定了位置 `p = locs[l-1]` 之后，数组上的每个下标只可能有三种情况：

- `i < p` … `f[l][i] == f[l-1][i]`(位于 p 的左边，保持不动，原样保留)
- `i == p` … `f[l][p] == f[l-1][p] op f[l-1][p+1]`(正好落在 p 上，就在这里把两个数算掉)
- `i > p` … `f[l][i] == f[l-1][i+1]`(位于 p 的右边，从左边的下一位挪过来)

In [1]:
from z3 import *
from helper.z3 import all_solutions

OP_SYMBOLS = '+-*'   # 0:+, 1:-, 2:*


def calc_op(x1, x2, op):
    """按 op 的编号选择加、减、乘"""
    return If(op == 0, x1 + x2, If(op == 1, x1 - x2, x1 * x2))


def calc_tree(f, locs, ops):
    """把每个阶段的计算整理成约束列表返回"""
    exprs = []
    for l in range(1, len(f)):
        prev = f[l - 1]
        for i in range(len(f[l])):
            exprs.extend([
                Implies(i < locs[l - 1], f[l][i] == prev[i]),
                Implies(i == locs[l - 1], f[l][i] == calc_op(prev[i], prev[i + 1], ops[l - 1])),
                Implies(i > locs[l - 1], f[l][i] == prev[i + 1]),
            ])
    return exprs

### 用 If 挑选运算符

```python
If(op == 0, x1 + x2, If(op == 1, x1 - x2, x1 * x2))
```

把 `If(条件, 为真时的值, 为假时的值)` 一层层嵌套起来，就表示在三个运算之间切换。

| `op` | 计算 |
|---|---|
| 0 | `x1 + x2` |
| 1 | `x1 - x2` |
| 2 | `x1 * x2` |

条件一多，嵌套的层数就跟着增加，因此选择的项目较多时，使用 `Function` 反而读起来更舒服；而这一章只有三个运算，`If` 已经够用。

## 还原成算式

Z3 交出来的只是一堆数，想让读者看到式子，还得自己动手还原，下面先看其中用到的工具。`Expr` 是一个很小的类，其中装着“左式、运算符、右式”三项，其 `output()` 顺着各阶段的步骤一路拼出字符串。

由于 `*` 或 `-` 的右边一旦出现 `+` 或 `-` 就会产生歧义，因此括号不能省略，否则 `1 + 2 * 3` 与 `(1 + 2) * 3` 就混成了一回事。

In [2]:
class Expr:
    """把计算树变成带括号的字符串"""
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

    def __str__(self):
        # 左侧：* 的左边是 + - 时要加括号
        if isinstance(self.left, Expr) and self.op == '*' and self.left.op in '+-':
            left = f'({self.left})'
        else:
            left = f'{self.left}'

        # 右侧：* 或 - 的右边是 + - 时要加括号
        if isinstance(self.right, Expr) and self.op in '*-' and self.right.op in '+-':
            right = f'({self.right})'
        else:
            right = f'{self.right}'

        return f'{left} {self.op} {right}'


def output(numbers, locs, ops):
    """从数的顺序和步骤生成算式字符串"""
    exprs = [str(x) for x in numbers]
    for loc, op in zip(locs, ops):
        exprs[loc] = Expr(exprs[loc], OP_SYMBOLS[op], exprs[loc + 1])
        del exprs[loc + 1]
    return str(exprs[0])

## 先用小例子试一下

拿 `1 2 3 4 5` 来，手写一套固定不变的步骤传进去，其中位置与运算符都事先指定。

- 第 1 段：位置 2 用 `+`(`3 + 4`)
- 第 2 段：位置 0 用 `*`(`1 * 2`)
- 第 3 段：位置 1 用 `-`(`7 - 5`)
- 第 4 段：位置 0 用 `+`(`2 + 2`)

注意，由于每一步都按计算树的结构合并，因此结果应当是 `1 * 2 + ((3 + 4) - 5) = 4`。

In [3]:
n = 5
f = [[Int(f'f{l}_{i}') for i in range(n - l)] for l in range(n)]

solver = Solver()
for i in range(n):
    solver.add(f[0][i] == i + 1)      # 一开始是 1, 2, 3, 4, 5

locs = [2, 0, 1, 0]                   # 合并哪个位置
ops = [0, 2, 1, 0]                    # +, *, -, +
solver.add(calc_tree(f, locs, ops))

assert solver.check() == sat
m = solver.model()

expr = output([i + 1 for i in range(n)], locs, ops)
print(expr, "=", eval(expr))

print("各阶段的值")
for level in range(n):
    print([m[f[level][i]] for i in range(n - level)])

1 * 2 + 3 + 4 - 5 = 4
各阶段的值
[1, 2, 3, 4, 5]
[1, 2, 7, 5]
[2, 7, 5]
[2, 2]
[4]


手算的结果对上了：`1 * 2 + 3 + 4 - 5 = 4`，把它当作计算树来看，就是 `1 * 2 + ((3 + 4) - 5)`。各阶段的值也一目了然，数一次少一个。

## 寻找凑成 99 的组合

这一次位置和运算符都不由我们决定，全部交给 Z3。数一共有 9 个，因此约束只有两条：位置 `locs[i]` 不小于 0 且小于 `9-i-1`，运算符落在 0〜2 之间。

In [4]:
n = 9
f = [[Int(f'f{l}_{i}') for i in range(n - l)] for l in range(n)]

solver = Solver()
for i in range(n):
    solver.add(f[0][i] == i + 1)          # 从 1 到 9

locs = IntVector('loc', n - 1)            # 各阶段合并的位置
ops = IntVector('op', n - 1)              # 各阶段的运算符

for i, loc in enumerate(locs):
    solver.add(0 <= loc, loc < n - i - 1)  # 到该阶段元素个数 - 1 为止

for op in ops:
    solver.add(0 <= op, op <= 2)           # 0:+, 1:-, 2:*

solver.add(calc_tree(f, locs, ops))
solver.add(f[-1][0] == 99)                 # 最后剩下的数是 99

print(solver.check())

m = solver.model()
print("各阶段的值")
for level in range(n):
    print([m[f[level][i]] for i in range(n - level)])

locs_used = [m[x].as_long() for x in locs]
ops_used = [m[x].as_long() for x in ops]
print("位置:", locs_used)
print("运算符:", [OP_SYMBOLS[x] for x in ops_used])

sat
各阶段的值
[1, 2, 3, 4, 5, 6, 7, 8, 9]
[1, 2, -1, 5, 6, 7, 8, 9]
[1, 2, -1, 5, 6, 7, 17]
[1, 2, -1, 5, 6, 119]
[1, 2, -1, 5, -113]
[1, 3, 5, -113]
[1, 15, -113]
[1, -98]
[99]
位置: [2, 6, 5, 4, 1, 1, 1, 0]
运算符: ['-', '+', '*', '-', '-', '*', '+', '-']


最后一步，把 Z3 给出的那串步骤还原成算式，下面就是一个例子。

In [5]:
expr = output(list(range(1, 10)), locs_used, ops_used)
print(expr, "=", eval(expr))

1 - ((2 - (3 - 4)) * 5 + 6 - 7 * (8 + 9)) = 99


例如某次运行得到的是下面这样的式子(换一次运行，式子可能就变了)。

```
1 - ((2 - (3 - 4)) * 5 + 6 - 7 * (8 + 9)) = 99
```

Z3 只保证给出满足条件的第一个解，因此形状每一回都不一样，可是无论哪一种，算下来都是 99。

## 小结

- 数列固定的时候，括号如何添加，都可以写成“合并相邻两个数”的一串操作
- 每个阶段准备一个数组，`calc_tree()` 用 `Implies` 把“保留、计算、挪位”三种情况写清楚
- 运算符编成整数，计算本身依靠 `If` 切换
- 找到的步骤送回计算树(`Expr`)，式子就可以读了
- 换成“凑出 24”那种连数的顺序都自由的问题，还要再补一条排列约束(见上一章)